In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")

### Loading cleaned dataset from previuos notebook

In [7]:
df = pd.read_csv(
    r"C:\Users\User\OneDrive\Desktop\Interactive-Tourism-Analytics-Platform\data\cleaned\hotel_bookings_cleaned.csv"
)

In [14]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_stay
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,Not Applicable,Not Applicable,0,Transient,0.0,0,0,Check-Out,2015-07-01,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,Not Applicable,Not Applicable,0,Transient,0.0,0,0,Check-Out,2015-07-01,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,Not Applicable,Not Applicable,0,Transient,75.0,0,0,Check-Out,2015-07-02,1
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,Not Applicable,0,Transient,75.0,0,0,Check-Out,2015-07-02,1
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,Not Applicable,0,Transient,98.0,0,1,Check-Out,2015-07-03,2


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  object        
 1   is_canceled                     119390 non-null  int64         
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  object        
 5   arrival_date_week_number        119390 non-null  int64         
 6   arrival_date_day_of_month       119390 non-null  int64         
 7   stays_in_weekend_nights         119390 non-null  int64         
 8   stays_in_week_nights            119390 non-null  int64         
 9   adults                          119390 non-null  int64         
 10  children                        119390 non-null  int64  

In [9]:
# Convert reservation_status_date from object to datetime
df["reservation_status_date"] = pd.to_datetime(
    df["reservation_status_date"]
)

### Feature 1: Total Stay

In [13]:
df["total_stay"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

In [15]:
df[
    [
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "total_stay"
    ]
].head()

,stays_in_weekend_nights,stays_in_week_nights,total_stay
0,0,0,0
1,0,0,0
2,0,1,1
3,0,1,1
4,0,2,2


### Feature 2: Total Guests

In [16]:
# Create total number of guests
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

In [17]:
df[
    [
        "adults",
        "children",
        "babies",
        "total_guests"
    ]
].head()

,adults,children,babies,total_guests
0,2,0,0,2
1,2,0,0,2
2,1,0,0,1
3,1,0,0,1
4,2,0,0,2


### Feature 3: Estimated Revenue

In [18]:
# Calculate estimated booking revenue
df["estimated_revenue"] = (
    df["adr"] * df["total_stay"]
).round(2)

In [19]:
df[
    [
        "adr",
        "total_stay",
        "estimated_revenue"
    ]
].head()

,adr,total_stay,estimated_revenue
0,0.0,0,0.0
1,0.0,0,0.0
2,75.0,1,75.0
3,75.0,1,75.0
4,98.0,2,196.0


### Feature 4: Season

### Why?

**Managers want to compare tourism demand by season rather than just month.**

In [20]:
season_map = {
    "December": "Winter",
    "January": "Winter",
    "February": "Winter",

    "March": "Spring",
    "April": "Spring",
    "May": "Spring",

    "June": "Summer",
    "July": "Summer",
    "August": "Summer",

    "September": "Autumn",
    "October": "Autumn",
    "November": "Autumn"
}

df["season"] = df["arrival_date_month"].map(season_map)

In [21]:
df[
    [
        "arrival_date_month",
        "season"
    ]
].head()

,arrival_date_month,season
0,July,Summer
1,July,Summer
2,July,Summer
3,July,Summer
4,July,Summer


### Feature 5: Family Booking

### Why?

*This lets us compare bookings made by families versus smaller groups.*

In [22]:
import numpy as np

df["family_booking"] = np.where(
    df["total_guests"] > 2,
    "Yes",
    "No"
)

In [23]:
df[
    [
        "total_guests",
        "family_booking"
    ]
].head()

,total_guests,family_booking
0,2,No
1,2,No
2,1,No
3,1,No
4,2,No


### Feature 6: Lead Time Category
### Question

**Do customers book at the last minute or well in advance?**

#### Instead of working with raw numbers like 67, 142, 5, we'll group them into business-friendly categories.

In [24]:
# Categorize booking lead time
df["lead_time_category"] = pd.cut(
    df["lead_time"],
    bins=[-1, 30, 90, df["lead_time"].max()],
    labels=["Short", "Medium", "Long"]
)

In [25]:
df[["lead_time", "lead_time_category"]].head()

,lead_time,lead_time_category
0,342,Long
1,737,Long
2,7,Short
3,13,Short
4,14,Short


### Feature 7: Booking Status

 ### Question

***is_canceled uses 0 and 1.***-

**For dashboards and presentations, Completed and Canceled are much easier to understand.**

In [26]:
df["booking_status"] = df["is_canceled"].map({
    0: "Completed",
    1: "Canceled"
})

In [27]:
df[["is_canceled", "booking_status"]].head()

,is_canceled,booking_status
0,0,Completed
1,0,Completed
2,0,Completed
3,0,Completed
4,0,Completed


### Feature 8: Guest Type

### Question

#### What type of travelers are booking?

- Solo
- Couple
- Family/Group

In [28]:
def guest_type(total):
    if total == 1:
        return "Solo"
    elif total == 2:
        return "Couple"
    else:
        return "Family/Group"

df["guest_type"] = df["total_guests"].apply(guest_type)

In [29]:
df[["total_guests", "guest_type"]].head()

,total_guests,guest_type
0,2,Couple
1,2,Couple
2,1,Solo
3,1,Solo
4,2,Couple


### Feature 9: Arrival Quarter
### Question

##### Businesses often report quarterly rather than monthly.?

In [30]:
quarter_map = {
    "January": "Q1",
    "February": "Q1",
    "March": "Q1",
    "April": "Q2",
    "May": "Q2",
    "June": "Q2",
    "July": "Q3",
    "August": "Q3",
    "September": "Q3",
    "October": "Q4",
    "November": "Q4",
    "December": "Q4"
}

df["arrival_quarter"] = df["arrival_date_month"].map(quarter_map)

In [31]:
df[["arrival_date_month", "arrival_quarter"]].head()

,arrival_date_month,arrival_quarter
0,July,Q3
1,July,Q3
2,July,Q3
3,July,Q3
4,July,Q3


### Feature 10: Long Stay
### Question

#### Hotels often distinguish between short and long stays.?

### We'll define a long stay as 7 nights or more.

In [32]:
df["long_stay"] = np.where(
    df["total_stay"] >= 7,
    "Yes",
    "No"
)

In [33]:
df[["total_stay", "long_stay"]].head()

,total_stay,long_stay
0,0,No
1,0,No
2,1,No
3,1,No
4,2,No


In [34]:
new_features = [
    "total_stay",
    "total_guests",
    "estimated_revenue",
    "season",
    "family_booking",
    "lead_time_category",
    "booking_status",
    "guest_type",
    "arrival_quarter",
    "long_stay"
]

df[new_features].head()

,total_stay,total_guests,estimated_revenue,season,family_booking,lead_time_category,booking_status,guest_type,arrival_quarter,long_stay
0,0,2,0.0,Summer,No,Long,Completed,Couple,Q3,No
1,0,2,0.0,Summer,No,Long,Completed,Couple,Q3,No
2,1,1,75.0,Summer,No,Short,Completed,Solo,Q3,No
3,1,1,75.0,Summer,No,Short,Completed,Solo,Q3,No
4,2,2,196.0,Summer,No,Short,Completed,Couple,Q3,No


In [35]:
df.to_csv("../data/processed/hotel_bookings_final.csv", index=False)

In [36]:
df.info()

df.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 42 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           119390 non-null  object        
 1   is_canceled                     119390 non-null  int64         
 2   lead_time                       119390 non-null  int64         
 3   arrival_date_year               119390 non-null  int64         
 4   arrival_date_month              119390 non-null  object        
 5   arrival_date_week_number        119390 non-null  int64         
 6   arrival_date_day_of_month       119390 non-null  int64         
 7   stays_in_weekend_nights         119390 non-null  int64         
 8   stays_in_week_nights            119390 non-null  int64         
 9   adults                          119390 non-null  int64         
 10  children                        119390 non-null  int64  

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_stay,total_guests,estimated_revenue,season,family_booking,lead_time_category,booking_status,guest_type,arrival_quarter,long_stay
count,119390,119390.000000,119390.000000,119390.000000,119390,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119390,119390,119390,119390,119390.000000,119390.000000,119390.000000,119390,119390,119390.000000,119390,119390,119390,119390.000000,119390,119390.000000,119390.000000,119390.000000,119390,119390,119390.000000,119390.000000,119390.000000,119390,119390,119390,119390,119390,119390,119390
unique,2,NaN,NaN,NaN,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,178,8,5,NaN,NaN,NaN,10,12,NaN,3,334,353,NaN,4,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,4,2,3,2,3,4,2
top,City Hotel,NaN,NaN,NaN,August,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BB,PRT,Online TA,TA/TO,NaN,NaN,NaN,A,A,NaN,No Deposit,9.0,Not Applicable,NaN,Transient,NaN,NaN,NaN,Check-Out,NaN,NaN,NaN,NaN,Summer,No,Long,Completed,Couple,Q3,No
freq,79330,NaN,NaN,NaN,13877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92310,48590,56477,97870,NaN,NaN,NaN,85994,74053,NaN,104641,31961,112593,NaN,89613,NaN,NaN,NaN,75166,NaN,NaN,NaN,NaN,37477,104812,51131,75166,82051,37046,105478
mean,NaN,0.370416,104.011416,2016.156554,NaN,27.165173,15.798241,0.927599,2.500302,1.856403,0.103886,0.007949,NaN,NaN,NaN,NaN,0.031912,0.087118,0.137097,NaN,NaN,0.221124,NaN,NaN,NaN,2.321149,NaN,101.831122,0.062518,0.571363,NaN,2016-07-30 00:24:47.883407104,3.427900,1.968239,357.848208,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,0.000000,0.000000,2015.000000,NaN,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,-6.380000,0.000000,0.000000,NaN,2014-10-17 00:00:00,0.000000,0.000000,-63.800000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,0.000000,18.000000,2016.000000,NaN,16.000000,8.000000,0.000000,1.000000,2.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,69.290000,0.000000,0.000000,NaN,2016-02-01 00:00:00,2.000000,2.000000,146.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,0.000000,69.000000,2016.000000,NaN,28.000000,16.000000,1.000000,2.000000,2.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,94.575000,0.000000,0.000000,NaN,2016-08-07 00:00:00,3.000000,2.000000,267.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,1.000000,160.000000,2017.000000,NaN,38.000000,23.000000,2.000000,3.000000,2.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,126.000000,0.000000,1.000000,NaN,2017-02-08 00:00:00,4.000000,2.000000,446.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,1.000000,737.000000,2017.000000,NaN,53.000000,31.000000,19.000000,50.000000,55.000000,10.000000,10.000000,NaN,NaN,NaN,NaN,1.000000,26.000000,72.000000,NaN,NaN,21.000000,NaN,NaN,NaN,391.000000,NaN,5400.000000,8.000000,5.000000,NaN,2017-09-14 00:00:00,69.000000,55.000000,7590.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
